# Phase 6: Training Loop

Goal of this phase: build the actual training loop, contrastive loss, optimizer, learning rate schedule, checkpointing, and confirm it actually works with a short trial before committing to a real run. Following the same rule as phase 5, all of the actual logic lives in src/training/, not in this notebook. This notebook records what I built, what the trial run caught, and where things stand before deciding on the real training settings.

## What I built

src/training/scheduler.py: cosine_lr_with_warmup, a small closure based scheduler that linearly ramps the learning rate up over a warmup period, then decays it following a cosine curve down to zero by the end of training. Same shape as the convention I found in open_clip_train's own reference scheduler back in phase 5's research.

src/training/train.py:
- clamp_logit_scale, clamping logit_scale to [0, ln(100)] after every optimizer step, the standard CLIP convention.
- train_one_epoch, one pass over the training data: forward pass under bf16 autocast, ClipLoss, backward, optional gradient clipping, optimizer step, then the logit_scale clamp. Logs progress every few steps (loss, time per step, peak memory) rather than staying silent for an entire epoch.
- evaluate, the same forward pass without gradients, over the validation set, used to get a validation loss per epoch.
- train, the actual driver: builds the model with the chosen freeze mode, builds the train and val Dataset/DataLoader from phase 4, builds the AdamW optimizer over only the trainable parameters, builds the scheduler, then loops over epochs, checkpointing whenever validation loss improves and stopping early if it does not improve for patience epochs in a row.

Defaults going in: partial fine-tuning, bf16 autocast, batch size 128 (from phase 5's AMP finding), learning rate 1e-5 (the CS231N BiomedCLIP fine-tuning precedent), weight decay 0.1, warmup 150 steps, gradient clipping at norm 1.0, checkpoint on best validation loss, patience 5 epochs.

## Why I ran a trial before a real run

This is a lot of brand new code (Dataset, DataLoader, model, freeze logic, ClipLoss, autocast, gradient clipping, logit_scale clamping, the scheduler, checkpointing) wired together for the first time. Cheaper to catch a bug on 20 steps than partway through a real multi hour run. I ran it as python -m style standalone execution, not live in this notebook, same reasoning as phase 5: a long lived process running many heavy steps in a row is where problems tend to actually show up, and a notebook kernel is not a reliable place to catch that.

## First trial attempt: caught a real memory problem

First trial: 1 epoch capped at 20 steps, batch size 128 (phase 5's recommended setting). This ran for over an hour before I checked on it, when it should have taken about a minute based on phase 5's own throughput numbers.

Checked nvidia-smi while it was still running: 100 percent GPU utilization, 5654 out of 6144 MiB used. Not a hang, it was genuinely computing, just very slowly, and using notably more memory than phase 5's isolated single step benchmark predicted for this exact config (5.04GB). My read: phase 5's benchmark measured exactly one step in a fresh process. Running 20 consecutive steps in the same process pushed memory higher than that single step number, into the same kind of throughput collapse phase 5 already found near the fp32 memory ceiling, just triggered here by sustained use rather than a single oversized batch.

Fix: dropped the default batch size in train() from 128 to 96, which phase 5 measured at a safer 4.00GB single step, leaving more real margin for this sustained-use effect. Also added per-step progress logging (loss, time per step, peak memory every 5 steps) to train_one_epoch, since the complete silence during that hour long stall is part of what made it hard to diagnose in the first place. That is a permanent improvement to the training loop itself, not just a one off debugging aid.

## Second trial attempt: the loss was completely wrong

Rerunning with batch size 96, the memory problem was gone: stable 4.24GB, about 1.1 seconds per step. But the loss values printed out were 1711, 1548, 1549, 1264. A properly scaled contrastive loss for a 96-way batch should start somewhere around ln(96), about 4.56, not four digits.

Traced it to encode_image and encode_text: open_clip's own model class defaults both to normalize=False, and I was calling them with no arguments in both train.py and benchmark.py. ClipLoss itself does not normalize internally, I confirmed this directly in its source back in phase 5, it expects the caller to hand it already normalized embeddings. So both files were computing the loss over raw, unnormalized embeddings, which have unbounded magnitude, producing meaningless logits and a meaningless loss.

I checked open_clip's own forward() and get_logits() methods to confirm this was really the issue rather than guessing: both explicitly call encode_image(image, normalize=True) and encode_text(text, normalize=True) internally. Fixed by adding normalize=True to both calls in train.py and benchmark.py.

Important distinction: this bug did not affect any of phase 5's memory or throughput numbers, since normalizing a vector is computationally trivial next to a full forward and backward pass through a transformer. It only affected whether the loss itself meant anything. If I had started a real training run without catching this, the model would have trained against a meaningless objective the whole time, with no crash and no obvious signal that anything was wrong beyond the loss looking unusually large, which I could easily have written off as normal early-training noise if I had not already had a rough idea of what to expect.

## Third attempt: clean

Same trial (1 epoch, 20 steps, batch 96), both fixes applied:

step 5   loss=4.0457  1.28s/step  peak_vram=4.24GB
step 10  loss=3.0277  1.27s/step  peak_vram=4.24GB
step 15  loss=3.1469  1.28s/step  peak_vram=4.24GB
step 20  loss=3.4050  1.26s/step  peak_vram=4.24GB
epoch 0  train_loss=3.5531  val_loss=3.4845  logit_scale=4.4452

Loss is now in the right range from the start (around ln(96) as expected) and already trending down within just 20 steps. Memory stable at 4.24GB, no sign of the earlier slowdown. train_loss and val_loss are close to each other, which is expected this early, nothing to read into yet given this was 20 steps, not a real epoch.

Checked the checkpoint too, not just the printed numbers: checkpoints/trial.pt is 783.7MB, matches the expected size for the full model's fp32 state dict (195.9 million parameters, saving the whole model including frozen weights since inference needs all of it, not just the trainable subset). Loaded it back and confirmed the epoch and val_loss inside match what was printed, and it holds 351 parameter tensors.

## What each training setting is, why I chose it, and how confident I actually am

Before running this for real I want a clear record of what every setting in train() actually is and why it has the value it has, since some of these are backed by real measurements on my own hardware, some are taken directly from a specific source, and some are my own reasoned compromise without a source I can point to. Worth being honest about which is which rather than presenting all of them as equally justified.

### Freeze mode: partial fine-tuning, last 2 of 12 layers per encoder

A neural network learns by adjusting its parameters based on gradients computed during backpropagation. Freezing a parameter means telling the optimizer never to update it, it stays exactly at its pretrained value for the whole run. This matters for two reasons: fewer trainable parameters means less GPU memory needed (fewer gradients and less optimizer state to store), and keeping a pretrained model's general-purpose layers frozen while only letting the later, more specialized layers adapt avoids overwriting useful pretrained knowledge with a small new dataset, sometimes called catastrophic forgetting.

Full fine-tuning (nothing frozen) needs about 9.87GB, measured directly in phase 5, more than my 6.44GB GPU has, so that option was ruled out on hardware grounds. Between freezing both encoders entirely and partially unfreezing them, I chose partial: freezing everything means my image and text representations never adapt to my dataset at all, they stay exactly as BiomedCLIP learned them from its own pretraining corpus. My dataset is a real domain shift from that, a specific hospital's equipment and reporting style, plus translated rather than natively English text, so I wanted real adaptation capacity, not just a linear re-projection of a fixed representation.

Unfreezing specifically the last 2 of 12 layers in each encoder follows the same principle Zhang et al. (2022), ConVIRT, uses for its own text encoder: they keep the pretrained weights for the first 6 layers of a BERT encoder and only fine-tune the last 6. Early layers of a pretrained network tend to encode general features, later layers are more task-specific. I chose a lighter touch than their 50 percent (2 of 12, about 17 percent), mainly because my memory budget is tighter than theirs was. This specific number, 2, is not derived from a formula, it is my own starting point within a principle I did find directly supported in the literature, and a real candidate to revisit if training underperforms.

Source: Zhang, Y., Jiang, H., Miura, Y., Manning, C.D., & Langlotz, C.P. (2022). Contrastive Learning of Medical Visual Representations from Paired Images and Text. PMLR 182:1-24.

### Precision: bfloat16 mixed precision

Standard training uses 32 bit floating point numbers. Mixed precision training does most of the computation in a lower precision format instead, here bfloat16, which uses half the memory per number and computes faster on GPUs with hardware support for it, which the RTX 3060 has.

Tested this directly rather than assuming it would help: roughly triples throughput (13ms per sample versus 41.79ms in fp32 at batch 64) and doubles the safe batch size ceiling (128 versus 64), with no real downside. bfloat16 has the same numeric range as fp32, just less precision, so unlike a different low precision format, float16, it does not need a GradScaler to avoid small gradients underflowing to zero. It is also the same precision BiomedCLIP's own pretraining used.

Source for the measurement: my own benchmarking, src/models/benchmark.py, phase 5. Source for BiomedCLIP's own use of bf16: Zhang, S. et al. (2023). BiomedCLIP: a multimodal biomedical foundation model pretrained from fifteen million scientific image-text pairs. arXiv:2303.00915, supplementary training details.

### Batch size: 96

In contrastive learning, batch size is not just a speed knob, it directly affects what the model learns from: every image in a batch is contrasted against every other text in that same batch as a negative example, so a bigger batch gives more negatives per step, generally a better training signal.

Phase 5's single step benchmark said 128 was safe (5.04GB). Phase 6's real multi-step trial run proved that number was not representative of sustained training: 20 consecutive steps pushed memory to 5.65GB, 92 percent of the GPU's total, and training slowed to roughly an hour for 20 steps instead of the expected minute. Lowered to 96, which phase 5 had already measured with more headroom (4.00GB single step), and the trial confirmed it holds stable at 4.24GB under real sustained use. The lesson: a one step benchmark is not the same test as an actual training loop, and I should not trust a memory number that was only ever tested once in isolation.

Source: my own benchmarking (src/models/benchmark.py) and the phase 6 trial run itself (src/training/train.py), not a literature value.

### Learning rate: 1e-5

The learning rate controls how large a step the optimizer takes in the direction the gradient points. Too large and training can be unstable or overwrite useful pretrained knowledge, too small and training barely moves.

This value is taken directly from a source, not derived from my own data: a Stanford CS231N course report, "Parameter-Efficient Fine-Tuning of BiomedCLIP for Diabetic Retinopathy" (2025), fine-tunes this exact BiomedCLIP checkpoint using AdamW at learning rate 1e-5. I am treating this as a directional reference point, not an authority, since it is a student project report and not peer reviewed. The broader reasoning that makes it plausible beyond just matching someone else's number: fine-tuning an already well-trained model conventionally uses a learning rate much smaller than pretraining from scratch would, BiomedCLIP's own pretraining used 5e-4, fifty times larger, since fine-tuning only needs to nudge already-good weights rather than learn them from nothing.

This is the setting I am least confident is actually correct for my data specifically, since it has not been tested against my own loss curves yet, only inherited from a related but different fine-tuning setup.

Source: Stanford CS231N course report (2025), "Parameter-Efficient Fine-Tuning of BiomedCLIP for Diabetic Retinopathy."

Comparison source: Zhang, S. et al. (2023), BiomedCLIP, arXiv:2303.00915, for the pretraining-scale learning rate.

### Weight decay: 0.1

Weight decay is a regularization technique, it pulls weights toward zero a little on every step, discouraging any single weight from growing very large, which helps reduce overfitting.

I do not have a specific source for 0.1, it is my own compromise between two references that both do not quite match my situation. BiomedCLIP's own pretraining and open_clip_train's own defaults use 0.2, calibrated for full scale training from scratch on millions of examples. ConVIRT, training substantial parts of its encoders from a weaker initialization, uses 1e-6, much lighter. Neither of those is really my situation, a small partial fine-tune of an already strong checkpoint on a comparatively small dataset, so I picked a middle value rather than copying either one directly. This is the setting with the weakest justification in the whole configuration and a real candidate for tuning if I have time.

Sources considered: Zhang, S. et al. (2023), BiomedCLIP, arXiv:2303.00915 (0.2); Zhang, Y. et al. (2022), ConVIRT, PMLR 182:1-24 (1e-6); open_clip_train/params.py, installed locally (0.2 default).

### Warmup: 150 steps

Rather than starting at the full learning rate immediately, I ramp it up gradually over the first 150 steps, then decay it following a cosine curve for the rest of training. Starting at full strength immediately risks a large, destabilizing update before the model has adjusted at all to the new data and optimizer state.

General fine-tuning literature I found suggests a much shorter warmup for fine-tuning specifically than for pretraining from scratch, on the order of 100 to 300 steps, or about 10 percent of total training, rather than the thousands of steps BiomedCLIP's own pretraining used (2000 steps, but that pretraining ran for 32 epochs at a batch size of 4096, a completely different scale). My own training has about 241 steps per epoch (23165 train examples divided by batch size 96), so 150 steps is roughly two thirds of one epoch, inside the range the literature suggested, not calculated by a specific formula for my exact setup.

Source: general fine-tuning literature (web search, mixed sources, no single paper to cite precisely) contrasted against BiomedCLIP's own pretraining warmup, Zhang, S. et al. (2023), arXiv:2303.00915.

### Gradient clipping: max norm 1.0

Gradient clipping rescales the gradient if its overall size exceeds a threshold, here 1.0, so a single unusually difficult batch cannot produce a huge update that destabilizes training.

This is a conventional default used broadly across deep learning, not something I tuned for this dataset specifically. open_clip_train supports gradient clipping as an optional setting but does not hardcode a default value of its own, 1.0 is simply the common choice I see used widely elsewhere and adopted here as cheap insurance, particularly relevant since I am unfreezing real transformer layers, not just small adapters, which has more room to produce a rough update early on.

Source: general convention, open_clip_train/train.py (supports the option, does not itself set a default value).

### Epoch budget: up to 20, early stopping after 5 without improvement

I set an upper bound of 20 epochs but do not expect training to actually run that long. Early stopping monitors validation loss after every epoch and halts training if it has not improved for a set number of epochs in a row, here 5, keeping whichever checkpoint had the best validation loss rather than whatever the model looked like at the very last step. This avoids having to guess the exact right number of epochs in advance and protects against overfitting on a dataset that is not huge.

Both 20 and 5 are conventional defaults from general ML practice, not numbers derived from anything specific to this dataset. The CS231N precedent trained for just 1 epoch, but that was an extremely lightweight LoRA and BitFit adapter fine-tune, a much smaller amount of trainable capacity than my partial fine-tune, which unfreezes real transformer layers and likely needs more epochs to actually converge.

Source: general convention, contrasted against the CS231N course report's 1 epoch for a lighter-weight adapter method.

## Summary and what's next

Built the training loop (src/training/scheduler.py, src/training/train.py) and validated it end to end with a short trial rather than trusting brand new code on a real run. That trial caught two real bugs before they could waste a full training run: batch 128 was not actually safe for sustained use despite passing phase 5's single-step benchmark, and the loss was being computed over unnormalized embeddings, which would have made any real training run meaningless without ever crashing or obviously failing.

Current defaults: partial fine-tuning, bf16 autocast, batch size 96, learning rate 1e-5, weight decay 0.1, warmup 150 steps, gradient clipping at norm 1.0, checkpoint on best validation loss, patience 5, up to 20 epochs.

What is not decided yet: whether these are the right settings for an actual full training run, specifically the epoch cap, the patience value, and whether 1e-5 is really the right learning rate for my data rather than just the closest literature precedent I could find. That is a real training run away, and worth deciding deliberately rather than just letting the current defaults run.

## The real run

With the trial validated and every setting above written down, ran train() for real, full epochs this time, no max_steps cap.

Epoch by epoch:

| epoch | train_loss | val_loss |
|---|---|---|
| 0 | 2.55 | 2.39 |
| 1 | 1.74 | 2.30 |
| 2 | 1.59 | 2.28 |
| 3 | 1.49 | 2.27 (best) |
| 4 | 1.40 | 2.28 |
| 5 | 1.33 | 2.29 |
| 6 | 1.25 | 2.31 |
| 7 | 1.20 | 2.32 |
| 8 | 1.15 | 2.36 |

Training stopped itself after epoch 8, early stopping triggered since validation loss had not improved for 5 straight epochs (4 through 8, all worse than epoch 3).

This is a textbook overfitting curve. Train loss drops steadily and continuously the entire time, 2.55 down to 1.15, never levels off. Validation loss improves through epoch 3, then starts climbing back up even as train loss keeps falling. Past epoch 3 the model is increasingly memorizing the training set rather than learning anything that generalizes to images and text it has not seen. Early stopping did exactly what it is supposed to do here: it kept training a few epochs past the peak to make sure it was not a temporary blip, then stopped and the checkpoint that actually got saved is epoch 3's, not epoch 8's, since the checkpointing logic only overwrites the saved file when validation loss actually improves.

Mechanically everything held up across all 9 epochs: stable 4.24GB the entire run, no memory creep, consistent time per step throughout (roughly 1.2 seconds), no repeat of the earlier slowdown. logit_scale moved only slightly over the run, 4.4438 down to 4.4369, nowhere near the ln(100) clamp ceiling, so the clamp itself was never actually exercised in this run.

Verified the saved checkpoint directly rather than just trusting the printed log: checkpoints/best.pt is 783.7MB, and loading it back confirms epoch 3 and val_loss 2.2726, matching exactly what train() printed.

## What the overfitting pattern actually tells me

The dataset converges fast, useful validation improvement is essentially over by epoch 3 or 4 given the current learning rate, weight decay, and how much of the model is unfrozen. That is a real, data-backed finding, not a guess, and it changes how I think about a few of the settings I picked mostly from precedent rather than from testing:

The epoch budget of 20 was far more generous than needed, the real limiting factor turned out to be overfitting, not running out of a fixed epoch count. Patience of 5 worked as intended, caught the overfitting a few epochs after it started rather than immediately on the first uptick, which is what patience is for.

Open question worth deciding next rather than assuming: is epoch 3's validation loss the best I can do, or would something that fights overfitting harder, for example stronger weight decay, or fewer trainable parameters via frozen_backbone instead of partial, push the validation loss lower before it turns around. I have not tested that yet, this run only tells me that partial fine-tuning at the current settings overfits by epoch 4, not what the ceiling is for this dataset in general.

## Correction from phase 9: the overfitting reading was based on the wrong signal

Going back over this after phase 9, the section above needs qualifying. It says that past epoch 3 the model is increasingly memorizing the training set rather than learning anything that generalizes. That conclusion came entirely from the validation loss curve turning upward, and it turns out validation loss is not a reliable stand in for retrieval quality here.

In phase 9 I re-ran this same configuration while measuring validation Recall@10 after every epoch alongside the loss. They disagree:

| epoch | train loss | val loss | val R@10 |
|---|---|---|---|
| 3 | 1.1713 | 2.1222 (loss best) | 0.4966 |
| 4 | 1.1112 | 2.1298 | 0.5059 |
| 5 | 1.0778 | 2.1391 | 0.5086 |
| 6 | 1.0595 | 2.1414 | 0.5137 (retrieval best) |
| 7 | 1.0524 | 2.1411 | 0.5108 |

Validation loss bottoms out at epoch 3 and rises for three straight epochs after it. Retrieval quality improves across all three of those epochs and does not peak until epoch 6. So the model was still getting better at the actual task for several epochs after the loss said it had started degrading.

Why they can come apart: the contrastive loss scores the whole similarity distribution within a batch and penalises overconfidence, while Recall@10 only asks whether a correct match lands somewhere in the top 10 of the entire split. A model can become more overconfident, which the loss punishes, while its ranking holds or improves, which is all Recall@10 measures.

What still stands from the section above: the loss curve really does turn at epoch 3, train loss really does keep falling monotonically, and early stopping really did behave as designed. What does not stand is the interpretation that the model is getting worse at retrieval past epoch 3. Selecting the checkpoint on validation loss cost roughly 1.7 points of Recall@10 versus selecting on retrieval directly. Phase 9 switched the selection metric accordingly, and notebooks/009-model-tuning.ipynb has the full reasoning.

## Is there an ideal validation loss

Wanted to actually understand what 2.2726 means before deciding whether it is good, rather than just trusting that a lower number is automatically better.

There is no absolute ideal value for a contrastive loss like this one, unlike something like classification accuracy where 100 percent is a clean ceiling. The loss is a function of batch size: for a batch of N, a completely untrained model with zero learned signal would score around ln(N), since with N-way classification and no signal, cross entropy over a uniform guess is ln(N). At my batch size of 96, that is ln(96), about 4.56. My val_loss of 2.27 is meaningfully below that, which does confirm the model is discriminating far above random chance, but there is no 0-is-perfect ceiling either. A loss near 0 would mean the model assigns near total certainty to every single pairing, which in practice signals severe overfitting rather than a good outcome.

So 2.2726 only really means something in relative comparisons: better or worse than a different epoch of the same run (which is exactly how I used it, to pick epoch 3 over epoch 8), or better or worse than a different configuration under the same batch size and setup. It is not comparable to some external good CLIP loss benchmark, and is not comparable across papers unless batch size and everything else matches exactly.

## So is epoch 3 actually good enough

Cannot answer that from the loss number alone, and that is not a gap in the methodology, it is just what a training loss can and cannot tell me. Validation loss is a proxy metric, used during training to guide checkpoint selection and early stopping. It is a convenient, cheap to compute stand in for what I actually care about, not the actual measure of success.

The real measure of success for this thesis is the downstream task: given a diagnosis, does the model retrieve the correct image. That is Recall@K, embedding clustering quality, and manual inspection of real query results, exactly what phase 7 is for. Loss and retrieval quality are correlated but not identical, two checkpoints with different val_loss could perform similarly on retrieval, or a loss improvement might not translate into meaningfully better retrieval. Not knowable which of those is true here until phase 7 actually runs.

Decision: evaluate the epoch 3 checkpoint for real in phase 7 before deciding whether to go back and tune against overfitting further. Tuning against val_loss now, before knowing whether val_loss differences even matter for retrieval, risks spending real training time optimizing the wrong thing. Nothing about evaluating first forecloses coming back to try frozen_backbone or a different weight decay afterward, if the retrieval results actually suggest it is worth it.

## How the model actually processes an image

Wanted to trace exactly what happens to one image between it being a file on disk and it becoming the vector that goes into the loss, rather than treating the model as a total black box.

Path and file selection (src/data/paths.py, src/data/dataset.py): for a given image id, resolve its folder, list the available png slices, randomly pick one, the phase 4 slice selection logic, deterministic when there is only one option, which is 95 percent of the time.

Loading: PIL opens the file. These pngs are already 224 by 224, 8 bit greyscale, medical windowed and aspect ratio padded by whatever the original RadiologyNET export process was, so at this point it is a single channel image, pixel values 0 to 255.

Transform pipeline, pulled directly from BiomedCLIP's own pretrained config rather than invented:
- Resize and crop: training images get a random resized crop, scale 0.9 to 1.0, a very mild crop to 90 to 100 percent of the image then resize back to 224 by 224, deliberately narrow so it does not remove clinically relevant content, just adds slight positional jitter. Validation and test images instead get a deterministic resize plus center crop, no randomness, so evaluation is repeatable.
- RGB conversion: the source images are single channel greyscale, but the vision transformer expects 3 channel RGB input, since it was originally trained on natural, photo style images. This step replicates the one grey channel into three identical channels, no new information added, purely a format match.
- Tensor conversion: pixel values go from the 0 to 255 integer range into a 0.0 to 1.0 floating point range.
- Normalization: each of the 3 channels gets shifted and rescaled using CLIP's own fixed mean and std values, so the input distribution matches what the pretrained model originally saw during its own training.

Batching: the DataLoader stacks 96 individually transformed [3, 224, 224] tensors into one [96, 3, 224, 224] batch.

The vision transformer itself, ViT-B/16, base size, 16 by 16 pixel patches:
- Patching: the 224 by 224 image is split into a grid of 16 by 16 pixel patches, 14 by 14 equals 196 patches total. Each patch, 768 raw pixel values, gets linearly projected into a 768 dimensional patch embedding vector. The image becomes a sequence of 196 vectors instead of a grid of pixels.
- Positional information: a learned positional embedding gets added to each patch vector so the model knows where in the image each patch came from, since transformers otherwise have no inherent sense of spatial order.
- 12 transformer blocks: each block runs self attention, every patch looks at every other patch and learns how much to weight information from each, this is how the model learns relationships between different regions of the image, followed by a small feed forward network. Blocks 0 to 9 are frozen, exactly as BiomedCLIP originally learned them. Blocks 10 and 11 are the ones actually being fine-tuned, the phase 5 decision.
- Pooling: after the 12 blocks, the 196 patch level vectors get condensed into one single vector representing the whole image.
- Projection head (visual.head, the small trainable layer): projects that vector into the shared embedding space that both images and text live in, same dimensionality as the text encoder's output, so they can be directly compared.
- Normalization: encode_image is called with normalize=True, dividing the vector by its own length, turning it into a unit vector. This is the step whose absence caused the loss bug found earlier in this phase, and it is what makes the later image-text dot product equal a proper cosine similarity, bounded between -1 and 1, which is what ClipLoss expects.

That final unit vector is the image's embedding, the thing compared against every text embedding in the batch to compute the contrastive loss, and the thing phase 7's retrieval metrics will actually operate on.

## Summary and what's next

Have a real trained checkpoint now, checkpoints/best.pt, epoch 3, val_loss 2.2726, saved automatically by the training loop's own checkpointing logic rather than picked by hand. The full pipeline, Dataset through DataLoader through model through training loop, ran end to end for the first time without crashing, and produced a sensible, interpretable result.

Next: phase 7, evaluation, actually seeing what this checkpoint is good for, retrieval metrics, embedding clustering, manual inspection of real query results. Also worth deciding, before or during that, whether to treat epoch 3 here as good enough to evaluate, or to first try to push past this overfitting point with a different regularization setup, since I now know overfitting is the real constraint here, not GPU memory or training time.